# Préparer le graphe PEMS08 — récupération et validation

Les fichiers `data/raw/distance.csv` et `data/raw/pems08.npz` ont été contrôlés : **0 octet chacun**. Ils ne sont pas lus. Une paire de données authentiques a été récupérée depuis le dépôt des auteurs ASTGCN, à une révision Git fixée, dans `data/source/pems08/`.

Le téléchargement a été exécuté séparément par `scripts/fetch_pems08.py --download`. Ce notebook ne fait aucune requête réseau et n’entraîne aucun modèle. Les deux fichiers téléchargés sont inspectés, comparés au CSV existant, et leur provenance est conservée avec URLs et SHA-256. Le remplacement des fichiers vides et l’export d’adjacence restent bloqués si le mapping n’est pas identifié de manière univoque.

**Constat à expliquer :** les 170 séries correspondent à leurs indices NPZ, mais 25 capteurs appartiennent à quatre groupes de séries entièrement identiques. Le CSV ne permet donc pas, à lui seul, de distinguer leurs IDs physiques. Aucune correspondance n’est inventée pour lever cette ambiguïté.

In [1]:
# Importer les outils de lecture et de validation, sans dépendance aux modèles.
from pathlib import Path
from itertools import permutations
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

racine = Path.cwd()
if not (racine / "data/raw/traffic_dataset.csv").is_file():
    racine = racine.parent
assert (racine / "data/raw/traffic_dataset.csv").is_file()

# Renseigner des chemins explicites si plusieurs versions existent ou si les fichiers sont hors projet.
DISTANCE_PATH = racine / "data/source/pems08/d5d40645fa45471f2b473fc49a29ec3f0a993c6d/distance.csv"
NPZ_PATH = racine / "data/source/pems08/d5d40645fa45471f2b473fc49a29ec3f0a993c6d/pems08.npz"
# Ce manifeste doit fournir une preuve vérifiable du mapping du graphe, pas une simple hypothèse.
PROVENANCE_PATH = racine / "data/source/pems08/validated_provenance.json"
problemes = []
validations = {}
statistiques = None
A = None
groupes_identiques = []
fichiers = sorted(p for p in racine.rglob("*") if p.is_file())
distances_candidates = [p for p in fichiers if p.name.lower() in {"distance.csv", "pems08.csv"}]
npz_candidates = [p for p in fichiers if p.suffix.lower() == ".npz"]
display(pd.DataFrame([{"chemin": str(p), "format": p.suffix, "taille_octets": p.stat().st_size}
                     for p in fichiers if p.suffix.lower() in {".csv", ".npz", ".npy", ".txt", ".json", ".pkl", ".zip"}]))
print("Fichiers de distances candidats :", [str(p) for p in distances_candidates])
print("Archives NPZ candidates :", [str(p) for p in npz_candidates])

def choisir(chemin_explicite, candidats, description):
    # Refuser de choisir arbitrairement entre plusieurs versions.
    if chemin_explicite is not None:
        chemin = Path(chemin_explicite)
        if chemin.is_file() and chemin.stat().st_size > 0:
            return chemin
        problemes.append(description + " : chemin introuvable : " + str(chemin))
    elif len(candidats) == 1:
        return candidats[0]
    else:
        problemes.append(description + (" : fichier absent." if not candidats else " : plusieurs candidats, chemin explicite requis."))
    return None

distance_path = choisir(DISTANCE_PATH, distances_candidates, "distance.csv PEMS08")
npz_path = choisir(NPZ_PATH, npz_candidates, "pems08.npz PEMS08")
def sha256(chemin):
    # Relever les empreintes locales sans les présenter comme une authentification officielle.
    with chemin.open("rb") as flux:
        return hashlib.file_digest(flux, "sha256").hexdigest()

trafic = pd.read_csv(racine / "data/raw/traffic_dataset.csv")
display(trafic.head(5))
print("CSV actuel :", trafic.shape, "| locations :", trafic.location.nunique(), "| timesteps :", trafic.timestep.nunique())

# Montrer les fichiers vides sans les charger ; consulter la provenance de la récupération.
display(pd.DataFrame([{"chemin": str(racine / "data/raw" / nom),
    "taille_octets": (racine / "data/raw" / nom).stat().st_size}
    for nom in ["distance.csv", "pems08.npz"]]))
recuperation = json.loads(PROVENANCE_PATH.read_text(encoding="utf-8"))
print("Dépôt :", recuperation["repository"], "| révision :", recuperation["commit"])
print("URLs :", {nom: info["url"] for nom, info in recuperation["files"].items()})
print("SHA-256 :", {nom: info["sha256"] for nom, info in recuperation["files"].items()})

,chemin,format,taille_octets
0,C:\Users\User\GoFlow\data\raw\distance.csv,.csv,0
1,C:\Users\User\GoFlow\data\raw\pems08.npz,.npz,0
2,C:\Users\User\GoFlow\data\raw\traffic_dataset.csv,.csv,81272927
3,C:\Users\User\GoFlow\data\source\pems08\d5d406...,.csv,4032
4,C:\Users\User\GoFlow\data\source\pems08\d5d406...,.npz,18525419
5,C:\Users\User\GoFlow\data\source\pems08\downlo...,.json,1809
6,C:\Users\User\GoFlow\data\source\pems08\valida...,.json,3762
7,C:\Users\User\GoFlow\results\model_benchmark\b...,.csv,1508
8,C:\Users\User\GoFlow\results\model_benchmark\l...,.csv,175
9,C:\Users\User\GoFlow\results\model_benchmark\l...,.csv,811


Fichiers de distances candidats : ['C:\\Users\\User\\GoFlow\\data\\raw\\distance.csv', 'C:\\Users\\User\\GoFlow\\data\\source\\pems08\\d5d40645fa45471f2b473fc49a29ec3f0a993c6d\\distance.csv']
Archives NPZ candidates : ['C:\\Users\\User\\GoFlow\\data\\raw\\pems08.npz', 'C:\\Users\\User\\GoFlow\\data\\source\\pems08\\d5d40645fa45471f2b473fc49a29ec3f0a993c6d\\pems08.npz']


,timestep,location,flow,occupy,speed
0,1,0,133.0,0.0603,65.8
1,1,1,210.0,0.0589,69.6
2,1,2,124.0,0.0358,65.8
3,1,3,145.0,0.0416,69.6
4,1,4,206.0,0.0493,69.4


CSV actuel : (3035520, 5) | locations : 170 | timesteps : 17856


,chemin,taille_octets
0,C:\Users\User\GoFlow\data\raw\distance.csv,0
1,C:\Users\User\GoFlow\data\raw\pems08.npz,0


Dépôt : Davidham3/ASTGCN-2019-mxnet | révision : d5d40645fa45471f2b473fc49a29ec3f0a993c6d
URLs : {'distance.csv': 'https://raw.githubusercontent.com/Davidham3/ASTGCN-2019-mxnet/d5d40645fa45471f2b473fc49a29ec3f0a993c6d/data/PEMS08/distance.csv', 'pems08.npz': 'https://raw.githubusercontent.com/Davidham3/ASTGCN-2019-mxnet/d5d40645fa45471f2b473fc49a29ec3f0a993c6d/data/PEMS08/pems08.npz'}
SHA-256 : {'distance.csv': '859d68e7583ed4e6016588eaaca494d3ece286dae361614c5c1833017f87eaa7', 'pems08.npz': 'e1d03ce74e9fb79149e6e7c37c680214822c4a285baa2f42278cacabcd25c075'}


## 1. Arêtes, identifiants et distances

Le fichier attendu comporte deux identifiants d’extrémités et une distance. Les noms de colonnes sont examinés ; les colonnes ne sont pas réaffectées uniquement par leur position. On vérifie les valeurs finies, les IDs entiers, les distances non négatives et les doublons. Deux distances contradictoires pour une même arête bloquent la préparation.

Si les IDs sont hors de 0..169, une table `graph_sensor_id,npz_index` est nécessaire. Même des IDs dans cette plage demandent une preuve documentaire qu’il s’agit des indices du tenseur associé.

In [2]:
aretes = None
if distance_path is None:
    print("Inspection des distances non exécutée : fichier non localisé.")
else:
    try:
        brut = pd.read_csv(distance_path)
        display(brut.head(10))
        print("Chemin :", distance_path, "| lignes :", len(brut), "| colonnes :", list(brut.columns))
        alias = {"from": "source", "source": "source", "source_sensor_id": "source", "src": "source",
                 "to": "destination", "target": "destination", "destination": "destination", "dst": "destination",
                 "target_sensor_id": "destination", "distance": "distance", "cost": "distance"}
        brut = brut.rename(columns={c: alias.get(str(c).strip().lower(), c) for c in brut.columns})
        assert not brut.columns.duplicated().any(), "Colonnes ambiguës après identification."
        assert {"source", "destination", "distance"}.issubset(brut.columns), "Schéma d’arêtes non reconnu."
        aretes = brut[["source", "destination", "distance"]].apply(pd.to_numeric, errors="raise")
        assert not aretes.empty
        assert np.isfinite(aretes.to_numpy()).all()
        assert (aretes[["source", "destination"]] % 1 == 0).all().all()
        aretes[["source", "destination"]] = aretes[["source", "destination"]].astype("int64")
        assert (aretes.distance >= 0).all(), "Distance négative."
        ids = set(aretes.source) | set(aretes.destination)
        print("IDs source min/max :", aretes.source.min(), aretes.source.max())
        print("IDs destination min/max :", aretes.destination.min(), aretes.destination.max())
        print("Nœuds distincts observés dans les arêtes :", len(ids), "| IDs :", sorted(ids))
        print("IDs hors 0..169 :", sorted(ids - set(range(170))))
        display(aretes.distance.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).to_frame())
        doublons = aretes.duplicated(["source", "destination"]).sum()
        print("Arêtes dupliquées :", int(doublons), "| boucles explicites :", int((aretes.source == aretes.destination).sum()))
        assert aretes.groupby(["source", "destination"]).distance.nunique().le(1).all(), "Distances contradictoires."
        aretes = aretes.drop_duplicates(["source", "destination"])
        validations["aretes"] = True
    except (ValueError, AssertionError, OSError) as erreur:
        aretes = None
        problemes.append("Arêtes invalides : " + str(erreur))

,from,to,cost
0,9,153,310.6
1,153,62,330.9
2,62,111,332.9
3,111,11,324.2
4,11,28,336.0
5,28,169,133.7
6,138,135,354.7
7,135,133,387.9
8,133,163,337.1
9,163,20,352.0


Chemin : C:\Users\User\GoFlow\data\source\pems08\d5d40645fa45471f2b473fc49a29ec3f0a993c6d\distance.csv | lignes : 295 | colonnes : ['from', 'to', 'cost']
IDs source min/max : 1 169
IDs destination min/max : 0 169
Nœuds distincts observés dans les arêtes : 170 | IDs : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 1

,distance
count,295.000000
mean,315.895254
std,216.686639
min,6.300000
1%,39.548000
25%,240.300000
50%,328.100000
75%,372.150000
99%,764.848000
max,3274.400000


Arêtes dupliquées : 18 | boucles explicites : 0


## 2. Tenseur NPZ et comparaison avec le CSV

Les clés et formes de **tous les tableaux** sont affichées. Les axes sont identifiés par leurs tailles (17 856, 170, 3), sans présumer leur position. L’ordre des trois variables est recherché parmi les six permutations possibles.

Les premières observations des capteurs **0, 1, 10, 50, 100, 169** sont affichées. Leurs séries complètes servent à identifier une permutation unique, puis la correspondance est vérifiée sur **tous les capteurs et tous les timesteps**, avec `rtol=1e-5`, `atol=1e-6` pour les écarts de représentation numérique. Ces tolérances sont fixées avant lecture ; elles ne servent pas à rapprocher des séries incompatibles.

In [3]:
tenseur = None
ordre_variables = None
comparaisons = None
if npz_path is None:
    print("Inspection du NPZ et comparaison des séries non exécutées : archive absente.")
else:
    try:
        tableaux = []
        with np.load(npz_path, allow_pickle=False) as archive:
            print("Chemin :", npz_path, "| clés :", archive.files)
            for cle in archive.files:
                tableau = archive[cle]
                print("Clé :", cle, "| dimensions :", tableau.shape, "| type :", tableau.dtype)
                print("Premières valeurs :", tableau.ravel()[:12])
                if tableau.ndim == 3 and sorted(tableau.shape) == [3, 170, 17856]:
                    axes = tuple(tableau.shape.index(taille) for taille in (17856, 170, 3))
                    tableaux.append((cle, axes, tableau.transpose(axes).copy()))
        assert len(tableaux) == 1, "Aucun tenseur unique compatible avec 17856 temps × 170 capteurs × 3 variables."
        cle_tenseur, axes_tenseur, tenseur = tableaux[0]
        print("Axes identifiés (temps, capteurs, variables) :", axes_tenseur)
        assert np.isfinite(tenseur).all()
        assert set(trafic.location) == set(range(170))
        assert not trafic.duplicated(["timestep", "location"]).any()
        assert sorted(trafic.timestep.unique()) == list(range(1, 17857))
        assert len(trafic) == 17856 * 170
        features = ["flow", "occupy", "speed"]
        csv_tenseur = trafic.sort_values(["timestep", "location"])[features].to_numpy().reshape(17856, 170, 3)
        capteurs = [0, 1, 10, 50, 100, 169]
        lignes = []
        for capteur in capteurs:
            for t in range(5):
                ligne = {"location": capteur, "timestep_csv": t + 1}
                ligne.update({f"NPZ_canal_{c}": tenseur[t, capteur, c] for c in range(3)})
                ligne.update({f"CSV_{f}": csv_tenseur[t, capteur, c] for c, f in enumerate(features)})
                lignes.append(ligne)
        display(pd.DataFrame(lignes))
        compatibles = [permutation for permutation in permutations(range(3)) if
            np.allclose(tenseur[:, capteurs, :][:, :, permutation], csv_tenseur[:, capteurs, :], rtol=1e-5, atol=1e-6)]
        assert len(compatibles) == 1, "Ordre des variables incompatible ou ambigu."
        ordre_variables = dict(zip(features, compatibles[0]))
        print("Ordre confirmé : feature CSV → canal NPZ :", ordre_variables)
        lignes = []
        for capteur in range(170):
            for feature, canal in ordre_variables.items():
                valeurs_csv = csv_tenseur[:, capteur, features.index(feature)]
                valeurs_npz = tenseur[:, capteur, canal]
                lignes.append({"location": capteur, "variable": feature, "canal_npz": canal,
                    "écart_max": float(np.max(np.abs(valeurs_npz - valeurs_csv))),
                    "correspondance": bool(np.allclose(valeurs_npz, valeurs_csv, rtol=1e-5, atol=1e-6))})
        comparaisons = pd.DataFrame(lignes)
        display(comparaisons[comparaisons.location.isin(capteurs)])
        assert comparaisons.correspondance.all(), "Au moins une série ne correspond pas à son index NPZ."
        # Une égalité de valeurs ne suffit pas à identifier un capteur aux séries dupliquées.
        validations["csv_npz_values"] = True
        classes = {}
        for capteur in range(170):
            empreinte = hashlib.sha256(np.ascontiguousarray(tenseur[:, capteur, :]).tobytes()).hexdigest()
            classes.setdefault(empreinte, []).append(capteur)
        groupes_identiques = [groupe for groupe in classes.values() if len(groupe) > 1]
        print("Séries distinctes :", len(classes), "| groupes identiques :", groupes_identiques)
        print("Séries numériquement compatibles pour les 170 indices ; identité non univoque dans ces groupes.")
        assert not groupes_identiques, "Mapping CSV/NPZ non univoque : preuve de l’ordre d’export nécessaire pour les séries identiques."
        validations["csv_npz"] = True
        print("Correspondance vérifiée sur les 170 capteurs : location = index NPZ ; timestep CSV = index temps NPZ + 1.")
    except (ValueError, AssertionError, OSError, KeyError) as erreur:
        problemes.append("Validation du mapping CSV/NPZ bloquée : " + str(erreur))

Chemin : C:\Users\User\GoFlow\data\source\pems08\d5d40645fa45471f2b473fc49a29ec3f0a993c6d\pems08.npz | clés : ['data']


Clé : data | dimensions : (17856, 170, 3) | type : float64
Premières valeurs : [1.33e+02 6.03e-02 6.58e+01 2.10e+02 5.89e-02 6.96e+01 1.24e+02 3.58e-02
 6.58e+01 1.45e+02 4.16e-02 6.96e+01]
Axes identifiés (temps, capteurs, variables) : (0, 1, 2)


,location,timestep_csv,NPZ_canal_0,NPZ_canal_1,NPZ_canal_2,CSV_flow,CSV_occupy,CSV_speed
0,0,1,133.0,0.0603,65.8,133.0,0.0603,65.8
1,0,2,114.0,0.0532,66.9,114.0,0.0532,66.9
2,0,3,140.0,0.0622,66.8,140.0,0.0622,66.8
3,0,4,106.0,0.0452,68.9,106.0,0.0452,68.9
4,0,5,117.0,0.0574,67.4,117.0,0.0574,67.4
5,1,1,210.0,0.0589,69.6,210.0,0.0589,69.6
6,1,2,185.0,0.0550,68.5,185.0,0.0550,68.5
7,1,3,171.0,0.0466,69.9,171.0,0.0466,69.9
8,1,4,174.0,0.0529,68.0,174.0,0.0529,68.0
9,1,5,176.0,0.0484,68.7,176.0,0.0484,68.7


Ordre confirmé : feature CSV → canal NPZ : {'flow': 0, 'occupy': 1, 'speed': 2}


,location,variable,canal_npz,écart_max,correspondance
0,0,flow,0,0.000000e+00,True
1,0,occupy,1,9.714451e-17,True
2,0,speed,2,0.000000e+00,True
3,1,flow,0,0.000000e+00,True
4,1,occupy,1,9.887924e-17,True
5,1,speed,2,0.000000e+00,True
30,10,flow,0,0.000000e+00,True
31,10,occupy,1,9.714451e-17,True
32,10,speed,2,0.000000e+00,True
150,50,flow,0,0.000000e+00,True


Séries distinctes : 149 | groupes identiques : [[48, 61, 80, 121, 122, 123, 141, 142], [50, 51, 74, 75, 90, 91, 115, 131], [55, 70, 73, 81, 83, 84, 159], [62, 109]]
Séries numériquement compatibles pour les 170 indices ; identité non univoque dans ces groupes.


## 3. Provenance et mapping des IDs du graphe

La [configuration des auteurs ASTGCN](https://github.com/Davidham3/ASTGCN-2019-mxnet/blob/master/configurations/PEMS08.conf) associe `distance.csv` et `pems08.npz` à PEMS08 et à 170 sommets. Leur [chargeur](https://github.com/Davidham3/ASTGCN-2019-mxnet/blob/master/lib/utils.py) utilise les deux premiers IDs comme indices de matrice, sans mapping intermédiaire. Cette documentation décrit la distribution publiée ; elle n’authentifie pas des fichiers locaux simplement parce qu’ils portent le même nom.

Pour autoriser l’export, renseigner `PROVENANCE_PATH` vers un manifeste réellement établi contenant : `dataset` (PEMS08), `distance_sha256`, `npz_sha256`, `source_url`, `evidence` (justification vérifiable de la provenance et du mapping), et `id_semantics` (`npz_index` si prouvé). Si les IDs sont physiques, ajouter `mapping_csv` vers une table **graph_sensor_id,npz_index**, documentée et couvrant les 170 capteurs. Ce notebook ne crée ni ce manifeste ni cette correspondance.

Les empreintes locales permettent de rattacher la preuve aux fichiers inspectés, mais ne constituent pas seules une preuve d’origine officielle. Ne pas utiliser un graphe METR-LA ou PEMS-BAY ni remplir les champs de preuve par une supposition.

In [4]:
mapping_graphe = None
provenance = None
if aretes is not None and validations.get("csv_npz_values", validations.get("csv_npz")):
    try:
        assert PROVENANCE_PATH is not None, "Preuve du mapping/provenance non fournie ; renseigner PROVENANCE_PATH après vérification documentaire."
        provenance_path = Path(PROVENANCE_PATH)
        provenance = json.loads(provenance_path.read_text(encoding="utf-8"))
        assert provenance["dataset"].upper().replace("-", "") == "PEMS08"
        assert provenance["distance_sha256"] == sha256(distance_path)
        assert provenance["npz_sha256"] == sha256(npz_path)
        assert provenance.get("source_url") and provenance.get("evidence"), "Source et justification manquantes."
        ids = set(aretes.source) | set(aretes.destination)
        if provenance["id_semantics"] == "npz_index":
            assert ids.issubset(set(range(170)))
            mapping_graphe = {i: i for i in range(170)}
        else:
            chemin_mapping = Path(provenance["mapping_csv"])
            if not chemin_mapping.is_absolute():
                chemin_mapping = provenance_path.parent / chemin_mapping
            table_mapping = pd.read_csv(chemin_mapping)
            display(table_mapping.head(10))
            assert {"graph_sensor_id", "npz_index"}.issubset(table_mapping.columns)
            assert len(table_mapping) == 170
            assert table_mapping.graph_sensor_id.is_unique and table_mapping.npz_index.is_unique
            assert set(table_mapping.npz_index) == set(range(170))
            mapping_graphe = dict(zip(table_mapping.graph_sensor_id, table_mapping.npz_index))
            assert ids.issubset(mapping_graphe)
        validations["mapping_graphe"] = True
        print("Preuve documentaire fournie :", provenance["source_url"], provenance["evidence"])
    except (ValueError, AssertionError, OSError, KeyError) as erreur:
        mapping_graphe = None
        problemes.append("Mapping du graphe non validé : " + str(erreur))
else:
    print("Mapping du graphe non évalué : arêtes ou comparaison CSV/NPZ non validées.")

Preuve documentaire fournie : https://github.com/Davidham3/ASTGCN-2019-mxnet/tree/d5d40645fa45471f2b473fc49a29ec3f0a993c6d/data/PEMS08 La configuration de cette révision associe ces deux fichiers à PEMS08 (170 sommets). Le chargeur des auteurs utilise les IDs entiers des arêtes directement comme indices A[i,j], sans remapping.


## 4. Stratégie de pondération et contrôles du graphe

**Méthode prévue, à exécuter après validation :** conserver les orientations et uniquement les arêtes explicitement fournies. Ne pas ajouter l’arête inverse, de voisins consécutifs ou de boucles artificielles. Si le fichier est symétrique, la matrice le sera naturellement ; sinon, le futur STGCN devra accepter un support dirigé ou documenter séparément une symétrisation.

Pour une distance d, utiliser **exp(−d² / σ²)** avec σ égal à l’écart-type (ddof=0) des distances des arêtes uniques. Ce choix de noyau et d’échelle suit le [générateur de graphe des auteurs DCRNN](https://github.com/liyaguang/DCRNN/blob/master/scripts/gen_adj_mx.py). C’est une méthode de pondération publiée, **pas une prétendue matrice officielle PEMS08**. Contrairement à son seuil de parcimonie, aucun seuil supplémentaire n’est appliqué ici : on conserve le support des arêtes fourni. σ non positif ou un poids numériquement nul bloque l’export plutôt que de choisir un autre paramètre arbitraire.

Les métriques incluent les arcs orientés, le degré entrant/sortant moyen, les forces pondérées, les nœuds isolés et la symétrie. La diagonale reflète seulement les boucles explicites. Une matrice dirigée ne doit pas être utilisée silencieusement avec un opérateur spectral exigeant un graphe symétrique.

In [5]:
if all(validations.get(cle, False) for cle in ["aretes", "csv_npz", "mapping_graphe"]):
    try:
        sigma = float(aretes.distance.to_numpy(dtype="float64").std(ddof=0))
        assert np.isfinite(sigma) and sigma > 0, "Écart-type des distances nul ou invalide."
        poids = np.exp(-np.square(aretes.distance.to_numpy(dtype="float64") / sigma))
        assert np.isfinite(poids).all() and (poids > 0).all(), "Poids non finis ou sous-flux numérique."
        A = np.zeros((170, 170), dtype="float64")
        sources = aretes.source.map(mapping_graphe).to_numpy(dtype="int64")
        destinations = aretes.destination.map(mapping_graphe).to_numpy(dtype="int64")
        A[sources, destinations] = poids
        assert A.shape == (170, 170) and np.isfinite(A).all()
        assert (A >= 0).all() and (A <= 1).all()
        assert np.count_nonzero(A) == len(aretes)
        support = A > 0
        hors_diagonale = support.copy()
        np.fill_diagonal(hors_diagonale, False)
        entrants = hors_diagonale.sum(axis=0)
        sortants = hors_diagonale.sum(axis=1)
        statistiques = {
            "shape": list(A.shape), "sigma": sigma, "arcs_total": int(support.sum()),
            "arcs_hors_diagonale": int(hors_diagonale.sum()), "boucles": int(np.trace(support)),
            "degre_sortant_moyen": float(sortants.mean()), "degre_entrant_moyen": float(entrants.mean()),
            "force_sortante_moyenne": float(A.sum(axis=1).mean()),
            "noeuds_isoles": np.flatnonzero((entrants + sortants) == 0).tolist(),
            "sans_sortie": np.flatnonzero(sortants == 0).tolist(),
            "sans_entree": np.flatnonzero(entrants == 0).tolist(),
            "support_symetrique": bool(np.array_equal(support, support.T)),
            "poids_symetriques": bool(np.allclose(A, A.T, rtol=1e-12, atol=1e-12)),
            "ecart_symetrie_max": float(np.abs(A - A.T).max()),
            "poids_min": float(poids.min()), "poids_max": float(poids.max()), "nan": int(np.isnan(A).sum())
        }
        display(pd.DataFrame([statistiques]))
        validations["adjacence"] = True
    except (ValueError, AssertionError) as erreur:
        A = None
        problemes.append("Construction du graphe bloquée : " + str(erreur))
else:
    print("Matrice non construite : prérequis non validés.")

Matrice non construite : prérequis non validés.


## 5. Export conditionnel et conclusion unique

L’export `data/processed/pems08_adjacency.npy` est autorisé seulement si les arêtes, toutes les séries CSV/NPZ, la preuve de mapping et les contrôles d’adjacence sont validés. Le fichier est relu pour vérifier son intégrité. Un Markdown décrit les entrées, la provenance, les comparaisons et la stratégie ; il indique explicitement les étapes non exécutées.

Si les fichiers manquent ou sont incompatibles, conclusion **C : STGCN à ne pas lancer**. Si les données authentiques sont récupérées et compatibles mais que les IDs du CSV restent indiscernables, conclusion **B : mapping encore incertain**, sans construction ni export de graphe. Le cas A concerne le support spatial validé ; le choix du futur opérateur STGCN, notamment sa compatibilité avec l’asymétrie éventuelle, reste à documenter.

In [6]:
# Enregistrer l’adjacence uniquement après tous les contrôles.
valide = all(validations.get(cle, False) for cle in ["aretes", "csv_npz", "mapping_graphe", "adjacence"])
sortie = racine / "data/processed/pems08_adjacency.npy"
if valide:
    assert not sortie.exists(), "Une adjacence existe déjà : la comparer avant de la remplacer."
    sortie.parent.mkdir(parents=True, exist_ok=True)
    np.save(sortie, A, allow_pickle=False)
    np.testing.assert_array_equal(np.load(sortie, allow_pickle=False), A)
    conclusion = "A. Graphe validé et prêt pour STGCN (support dirigé à respecter si non symétrique)."
elif validations.get("csv_npz_values") and groupes_identiques:
    conclusion = "B. Vrais fichiers PEMS08 récupérés mais mapping encore incertain pour les séries identiques."
else:
    conclusion = "C. Données nécessaires absentes, incompatibles ou non validées : STGCN à ne pas lancer."

# Produire un rapport même lorsque les données ne sont pas disponibles.
lignes = ["# Préparation du graphe PEMS08", "", "## Conclusion", "", conclusion, "",
    "## Entrées et provenance", "", f"- CSV : `{racine / 'data/raw/traffic_dataset.csv'}`.",
    f"- Fichier d’arêtes localisé : `{distance_path}`.", f"- NPZ localisé : `{npz_path}`.",
    "- Les noms de fichiers et les 170 locations ne prouvent pas l’origine officielle ni le mapping."]
for chemin in [distance_path, npz_path]:
    if chemin is not None:
        lignes.append(f"- SHA-256 local de `{chemin}` : `{sha256(chemin)}`.")
lignes += ["", "## Vérifications", "", "```json", json.dumps(validations, ensure_ascii=False, indent=2), "```",
    "", "## Mapping", "", "Ordre des variables confirmé : " + (str(ordre_variables) if validations.get("csv_npz_values") else "non évalué ou non validé"),
    "Correspondance location/index NPZ : " + ("vérifiée sur toutes les séries" if validations.get("csv_npz") else "valeurs compatibles mais IDs ambigus dans quatre groupes" if validations.get("csv_npz_values") else "non confirmée"),
    "Preuve du mapping des IDs du graphe : " + (json.dumps(provenance, ensure_ascii=False) if validations.get("mapping_graphe") else "non disponible ou non validée"),
    "", "## Méthode de création", "",
    "Méthode prévue : orientations du fichier conservées, poids exp(-d²/sigma²), sigma = écart-type des distances uniques (ddof=0), aucun seuil supplémentaire, aucune arête ajoutée.",
    "Méthode exécutée : " + ("oui" if valide else "non : aucun graphe construit/sauvegardé"),
    "Référence du noyau : https://github.com/liyaguang/DCRNN/blob/master/scripts/gen_adj_mx.py", "",
    "## Statistiques mesurées", "", "```json", json.dumps(statistiques, ensure_ascii=False, indent=2), "```",
    "", "## Blocages", ""]
lignes += ["", "Groupes de séries identiques : `" + str(groupes_identiques) + "`.",
    "Séries comparées : 170 capteurs × 17856 timesteps × 3 variables ; détail dans pems08_series_comparison.csv.",
    "Ordre réel des canaux : flow, occupy, speed. La numérotation du graphe vise les indices NPZ (chargeur des auteurs vérifié).",
    "Originaux téléchargés conservés dans data/source/pems08/ ; fichiers vides de data/raw/ non remplacés.",
    "Pour lever l’ambiguïté : obtenir le code ou manifeste de conversion qui prouve que location a été défini comme l’index capteur du NPZ, sans permutation."]
lignes += ["- " + probleme for probleme in problemes] or ["Aucun."]
lignes += ["", "## Fichiers manquants ou preuves à obtenir", "",
    "Le couple PEMS08 `distance.csv` / `pems08.npz`, puis la documentation prouvant le sens des IDs et leur correspondance avec le tenseur. Si les IDs sont physiques, ajouter une table documentée graph_sensor_id,npz_index.",
    "Configuration des auteurs : https://github.com/Davidham3/ASTGCN-2019-mxnet/blob/master/configurations/PEMS08.conf", "",
    "Aucun téléchargement, aucun modèle entraîné, aucune relation spatiale déduite des IDs consécutifs."]
rapport = racine / "results/spatial_audit/graph_preparation.md"
rapport.parent.mkdir(parents=True, exist_ok=True)
rapport.write_text("\n".join(lignes) + "\n", encoding="utf-8")
display(Markdown("### Conclusion\n\n**" + conclusion + "**\n\n" + "\n".join("- " + p for p in problemes)))
print("Rapport :", rapport)
print("Adjacence sauvegardée :", valide)

### Conclusion

**B. Vrais fichiers PEMS08 récupérés mais mapping encore incertain pour les séries identiques.**

- Validation du mapping CSV/NPZ bloquée : Mapping CSV/NPZ non univoque : preuve de l’ordre d’export nécessaire pour les séries identiques.

Rapport : C:\Users\User\GoFlow\results\spatial_audit\graph_preparation.md
Adjacence sauvegardée : False


## Résultat de la récupération

**B. Vrais fichiers PEMS08 récupérés mais mapping encore incertain.**

La clé du NPZ est `data`, de shape **(17856, 170, 3)** : temps, capteurs, variables. L’ordre réel est **flow, occupy, speed**, déterminé par comparaison des données. Le fichier spatial contient **295 lignes**, colonnes **from, to, cost**, et couvre les **170 IDs de 0 à 169**. Les distances sont les valeurs publiées ; leur unité n’est pas supposée. Les doublons d’arêtes sont contrôlés et présentés dans les cellules précédentes.

Sur les 170 capteurs, les valeurs correspondent au même index, avec seulement des écarts de représentation décimale pour occupy. Cependant, quatre groupes comprennent **25 capteurs aux séries intégralement identiques** : 145 capteurs sont distinguables individuellement par leurs données. Pour les autres, une permutation au sein du groupe est indétectable à partir du CSV.

Il manque le **code ou manifeste de conversion ayant créé traffic_dataset.csv**, prouvant que `location` a été attribué directement à l’index capteur du NPZ. Les fichiers téléchargés sont conservés dans `data/source/pems08/`, avec URLs, révision et SHA-256. Les deux fichiers vides dans `data/raw/` restent inchangés. Aucun graphe n’est exporté et aucun STGCN n’est lancé.

In [7]:
# Afficher les résultats réellement mesurés lors de la récupération.
resume_recuperation = json.loads((racine / "results/spatial_audit/pems08_retrieval_validation.json").read_text(encoding="utf-8"))
display(pd.DataFrame([resume_recuperation]))
apercu = pd.read_csv(racine / "results/spatial_audit/pems08_comparison_preview.csv")
display(apercu)
print("Les fichiers vides n’ont pas été remplacés ; aucune adjacence n’a été sauvegardée.")

,conclusion,npz_keys_shapes,npz_key,axes_time_sensor_feature,feature_to_npz_channel,csv_sha256,rtol,atol,all_170_full_series_match,unique_sensor_series,...,duplicate_sensor_groups,uniquely_identifiable_sensor_count,max_absolute_error,edge_columns,edge_count,distinct_edge_nodes,id_min,id_max,nodes_absent_from_edges,distance_statistics
0,B. Vrais fichiers PEMS08 récupérés mais mappin...,"{'data': [17856, 170, 3]}",data,"[0, 1, 2]","{'flow': 0, 'occupy': 1, 'speed': 2}",7bcc4a2161229a4b5187bd727c1e63f989aa193301d677...,0.00001,0.000001,True,False,...,"[[48, 61, 80, 121, 122, 123, 141, 142], [50, 5...",145,9.887924e-17,"[from, to, cost]",295,170,0,169,[],"{'count': 295.0, 'mean': 315.8952542372881, 's..."


,location,timestep,npz_channel_0,npz_channel_1,npz_channel_2,csv_flow,csv_occupy,csv_speed
0,0,1,133.0,0.0603,65.8,133.0,0.0603,65.8
1,0,2,114.0,0.0532,66.9,114.0,0.0532,66.9
2,0,3,140.0,0.0622,66.8,140.0,0.0622,66.8
3,0,4,106.0,0.0452,68.9,106.0,0.0452,68.9
4,0,5,117.0,0.0574,67.4,117.0,0.0574,67.4
5,1,1,210.0,0.0589,69.6,210.0,0.0589,69.6
6,1,2,185.0,0.0550,68.5,185.0,0.0550,68.5
7,1,3,171.0,0.0466,69.9,171.0,0.0466,69.9
8,1,4,174.0,0.0529,68.0,174.0,0.0529,68.0
9,1,5,176.0,0.0484,68.7,176.0,0.0484,68.7


Les fichiers vides n’ont pas été remplacés ; aucune adjacence n’a été sauvegardée.
